# Codebase-to-Wiki Q&A — on Mistral

Turn a codebase and its operational logs into a cited, regenerable wiki that answers install and operations questions.

The running example is **Robot Ross**: an autonomous robotic arm that records every action in a structured JSONL ledger. This notebook shows the full pipeline:

1. **Generate** — run `ledger_to_md.py` to compile JSONL logs → Markdown wiki page
2. **Index** — upload the corpus to a Mistral Document Library
3. **Analyse** — use Structured Outputs to extract typed metrics from a log slice
4. **Query** — ask questions; every answer is cited to a source document

Only `MISTRAL_API_KEY` is required for steps 2–4. Step 1 runs entirely offline.

## Step 1 — Generate the wiki from logs

`ledger_to_md.py` reads the JSONL operational ledger and compiles it into a Markdown wiki page. This is the step that makes the corpus *generated* rather than hand-written — re-run it on fresh logs and the wiki reflects the current system state automatically.

In [ ]:
from pathlib import Path
import subprocess, sys

def _find_root():
    """Walk up from CWD until we find the project root (contains tools/ledger_to_md.py)."""
    here = Path.cwd()
    for p in [here, here.parent, here.parent.parent]:
        if (p / "tools" / "ledger_to_md.py").exists():
            return p
    # Colab: cloned into cookbook root
    colab = here / "third_party" / "automated-technical-file"
    if (colab / "tools" / "ledger_to_md.py").exists():
        return colab
    raise FileNotFoundError("Cannot locate project root (tools/ledger_to_md.py not found)")

root = _find_root()

ledger   = root / "artifacts" / "ledger" / "sample_events.jsonl"
wiki_out = root / "artifacts" / "wiki"   / "sample_run_summary.md"

subprocess.run(
    [sys.executable, str(root / "tools" / "ledger_to_md.py"),
     "--input",  str(ledger),
     "--output", str(wiki_out),
     "--title",  "Sample Robot Run"],
    check=True,
)

print(f"Generated: {wiki_out}")
print()
print(wiki_out.read_text(encoding="utf-8")[:1200])

## Step 2 — Upload corpus to Mistral Document Library

The wiki pages (Overview, Bidding Rules, Hardware Interface, and the run summary just generated) are uploaded to a Mistral Document Library. Mistral chunks and indexes them; Citations will ground every Q&A answer in a specific document and section.

Set `MISTRAL_API_KEY` before running. Set `BACKEND=local` to skip the upload and use the local Ministral fallback instead.

In [ ]:
import os, time
from pathlib import Path

MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY")
BACKEND = os.environ.get("BACKEND", "auto").lower()

MODEL_HOSTED = "mistral-large-latest"
MODEL_LOCAL  = "ministral-3b-latest"

def _find_root():
    here = Path.cwd()
    for p in [here, here.parent, here.parent.parent]:
        if (p / "tools" / "ledger_to_md.py").exists():
            return p
    colab = here / "third_party" / "automated-technical-file"
    if (colab / "tools" / "ledger_to_md.py").exists():
        return colab
    raise FileNotFoundError("Cannot locate project root")

root = _find_root()

WIKI_CORPUS_FILES = [
    "notebooks/Overview.md",
    "notebooks/Topics/BiddingRules.md",
    "notebooks/Subsystems/HardwareInterface.md",
    "artifacts/wiki/sample_run_summary.md",
]

# Accept an existing library ID from env to skip re-uploading on repeat runs.
LIBRARY_ID = os.environ.get("MISTRAL_LIBRARY_ID")

def _upload_with_retry(client, library_id, file_name, content, max_retries=4):
    for attempt in range(max_retries):
        try:
            client.beta.libraries.documents.upload(
                library_id=library_id,
                file={"file_name": file_name, "content": content},
            )
            return
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 5 * (2 ** attempt)  # 5s, 10s, 20s, 40s
                print(f"  Rate limited — retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

try:
    from mistralai import Mistral
except ImportError:
    Mistral = None
    print("ERROR: mistralai not installed. Run: pip install mistralai")

if Mistral and BACKEND in ("auto", "hosted") and MISTRAL_API_KEY:
    client = Mistral(api_key=MISTRAL_API_KEY)
    if LIBRARY_ID:
        print(f"Reusing existing library: {LIBRARY_ID}")
    else:
        library = client.beta.libraries.create(
            name="RobotRoss Wiki Demo",
            description="Corpus for codebase-to-wiki Q&A notebook",
        )
        LIBRARY_ID = library.id
        for rel in WIKI_CORPUS_FILES:
            path = root / rel
            if not path.exists():
                print(f"WARNING: not found — {path}")
                continue
            with open(path, "rb") as f:
                _upload_with_retry(client, LIBRARY_ID, path.name, f.read())
            print(f"Uploaded: {rel}")
    print(f"\nLibrary ID: {LIBRARY_ID}")
elif BACKEND == "local":
    print("Local backend selected — Document Library step skipped.")
    print("Cited Q&A will use the corpus-only runtime_adapter fallback.")
else:
    print("Set MISTRAL_API_KEY (and optionally BACKEND=hosted) to build the library.")

## Step 3 — Structured log analysis

The same JSONL ledger feeds a second path: Mistral's Structured Outputs (JSON schema mode) extracts typed metrics and anomaly flags from a run slice. This is useful for dashboards, alerting, or feeding downstream tooling that needs machine-readable data rather than prose.

In [ ]:
import json, os
from pathlib import Path

MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY")
MODEL_HOSTED = "mistral-large-latest"  # repeated for standalone cell use

def _find_root():
    here = Path.cwd()
    for p in [here, here.parent, here.parent.parent]:
        if (p / "artifacts" / "ledger" / "sample_events.jsonl").exists():
            return p
    colab = here / "third_party" / "automated-technical-file"
    if (colab / "artifacts" / "ledger" / "sample_events.jsonl").exists():
        return colab
    raise FileNotFoundError("Cannot locate project root")

root = _find_root()
LEDGER_PATH = root / "artifacts" / "ledger" / "sample_events.jsonl"

LOG_ANALYSIS_SCHEMA = {
    "type": "object",
    "properties": {
        "total_events":      {"type": "integer"},
        "job_count":         {"type": "integer"},
        "error_count":       {"type": "integer"},
        "duration_seconds":  {"type": "number"},
        "anomalies":         {"type": "array", "items": {"type": "string"}},
        "recommendations":   {"type": "array", "items": {"type": "string"}},
    },
    "required": ["total_events", "job_count", "error_count", "anomalies", "recommendations"],
}

events = [json.loads(l) for l in LEDGER_PATH.read_text().splitlines() if l.strip()]
ledger_text = json.dumps(events[:30], indent=2)

PROMPT = (
    "Analyse this operational log slice from an autonomous robotic system.\n"
    "Return a JSON object: total_events, job_count, error_count, duration_seconds, "
    "anomalies (list), recommendations (list).\n\nLog slice:\n"
    + ledger_text
)

if MISTRAL_API_KEY:
    try:
        from mistralai import Mistral
        client   = Mistral(api_key=MISTRAL_API_KEY)
        response = client.chat.complete(
            model=MODEL_HOSTED,
            messages=[{"role": "user", "content": PROMPT}],
            response_format={"type": "json_object"},
        )
        result = json.loads(response.choices[0].message.content)
        print(json.dumps(result, indent=2))
    except Exception as e:
        print(f"Structured Outputs call failed: {e}")
else:
    print("Set MISTRAL_API_KEY to run Structured Outputs analysis.")
    print("Expected output shape:", json.dumps({k: "..." for k in LOG_ANALYSIS_SCHEMA["required"]}, indent=2))

## Step 4 — Cited Q&A

Questions are answered by a Mistral agent configured with the Document Library tool. Every answer includes Citations — reference IDs that point back to the exact wiki page and section used, so you can verify any claim against the original log or source file.

In [ ]:
import os, json, httpx

MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY")
MODEL_HOSTED = "mistral-large-latest"  # repeated for standalone cell use

DEMO_QUESTIONS = [
    "What is the bidding rule for the Wall of Fame?",
    "What calibration does the robot arm require at startup?",
    "Why might the last run have paused mid-job?",
]

# SDK 1.10.0: agents.complete parses the response as ChatCompletionResponse, which requires
# choices[0].message (singular). The agents endpoint with document_library tools returns
# choices[0].messages (list of DeltaMessage). Using httpx directly to avoid the model mismatch.
def ask_with_citations(api_key, agent_id, question):
    with httpx.Client(timeout=60.0) as http:
        resp = http.post(
            "https://api.mistral.ai/v1/agents/completions",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
            },
            json={
                "agent_id": agent_id,
                "messages": [{"role": "user", "content": question}],
            },
        )
        resp.raise_for_status()
    data = resp.json()
    choice = data["choices"][0]
    msgs = choice.get("messages", [])
    parts, sources = [], []
    for msg in msgs:
        if msg.get("role") == "assistant":
            content = msg.get("content", "")
            if isinstance(content, str) and content:
                parts.append(content)
            elif isinstance(content, list):
                for chunk in content:
                    if chunk.get("type") == "text" and chunk.get("text"):
                        parts.append(chunk["text"])
                    elif chunk.get("type") == "reference":
                        sources.extend(chunk.get("reference_ids", []))
    return "".join(parts), sources

if MISTRAL_API_KEY and LIBRARY_ID:
    from mistralai import Mistral
    client = Mistral(api_key=MISTRAL_API_KEY)

    agent = client.beta.agents.create(
        name="Wiki Q&A",
        model=MODEL_HOSTED,
        instructions="Answer questions grounded in the provided documents. Always cite sources.",
        tools=[{"type": "document_library", "library_ids": [LIBRARY_ID]}],
    )

    for q in DEMO_QUESTIONS:
        answer, sources = ask_with_citations(MISTRAL_API_KEY, agent.id, q)
        print(f"Q: {q}")
        print(f"A: {answer}")
        if sources:
            print(f"   Sources: {sources}")
        print()

elif not MISTRAL_API_KEY:
    print("Set MISTRAL_API_KEY and run Step 2 first to build the Document Library.")
else:
    print("LIBRARY_ID not set — run Step 2 first.")

## Step 5 (optional) — Voice loop with Voxtral

Voxtral Transcribe 2 converts a spoken question to text, the text goes through Step 4 exactly as typed, and Voxtral TTS speaks the cited answer back. This cell is a stub — wire `tools/voice/listen.py` and `tools/voice/speak.py` to the `ask_with_citations` function above to run it end-to-end.

No Whisper, no other providers. Full Mistral stack.